## AI4Climate ML tutorial - Training in PyTorch
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-02-10
* © British Crown Copyright 2017-2065, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

In this notebook, we will use the previously prepared tabular dataset to predict climate zones. We will train on different eras to see how well the results generalise with climate change. 

### Prerequisites 
what background information is needed to go through the notebook
- Same as previouis notebooks
- Have completed training pipeline, inference and evaluation notebooks.

### Learning outcomes from completing the notebook

- Understand how to build a pipeline using pytorch
- Understand the core pytorch concepts and classes
- Understand bho to manage experiments using ML Flow

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)

## Setup
First we start by loading the data we have prepared previously, and other set up elements

### Imports

In [3]:
import pathlib
import os
import datetime
import json

In [74]:
import numpy 
import pandas

In [5]:
import matplotlib
import matplotlib.pyplot

In [6]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


In [10]:
import mlflow

In [11]:
import torch

## Load and prepare data 
We will now load the dataset and do the usual data prep steps, like train/test split and normalisation.


#### Dataset parameters

In [19]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/nopw/j04/mohc_shared/dscop/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer, ver

In [20]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'climate_zones'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'climate_zones'
    return root_path

In [21]:
current_platform = tutorial_config['platform']

In [22]:
current_platform

'jasmin'

In [23]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones')

In [24]:
ml_ready_dir = root_data_dir / 'ml_ready'
print(ml_ready_dir.is_dir())
ml_ready_dir

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready')

In [28]:
resolutions_dict = {float(k1): v1 for k1,v1 in tutorial_config['resolutions_names'].items()}

dataset_prefix_dict = tutorial_config['dataset_prefix']

format_str = 'nc'
historic_scenario_str = 'historic'

future_scenario_list = tutorial_config['future_scenarios']
historic_scenario_list = tutorial_config['historic_scenarios']

time_periods = { 
    (1901,1930): historic_scenario_list, 
    (1931,1960): historic_scenario_list,
    (1961,1990): historic_scenario_list,
    (1991,2020): historic_scenario_list,
    (2041,2070): future_scenario_list,
    (2071,2099): future_scenario_list,
}


In [29]:
fname_template = tutorial_config['fname_template']
time_dir_template = tutorial_config['time_dir_template']
ml_ready_fname_template = tutorial_config['csv_out_template']

#### Load data for training

In [30]:
current_res = 1.0

In [31]:
mlready_data_path = ml_ready_dir / ml_ready_fname_template.format(resolution=resolutions_dict[current_res])
print(mlready_data_path.is_file())
mlready_data_path

True


PosixPath('/gws/nopw/j04/mohc_shared/dscop/climate_zones/ml_ready/climate_zones_1p0.csv')

In [32]:
zones_df = pandas.read_csv(mlready_data_path)

We have two options for the target. We have the full 30 class climate subgroups of the Koppen-Geiger classification data from the original dataset. We also have the processed five class climate group target, which presents an easier target for our classification algorithm to predict.

#### Selecting features

In [33]:
predictors_dict = {
    'precip_mean': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'mean' in c1],
    'precip_std': [c1 for c1 in zones_df.columns if 'precipitation' in c1 and 'std' in c1],
    'temp_mean': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'mean' in c1],
    'temp_std': [c1 for c1 in zones_df.columns if 'air_temperature' in c1 and 'std' in c1],
}


In [34]:
target_var = 'climate_group' # 5 classes
# target_var = 'climate_subgroup # 30 classes


In [35]:
predictors = predictors_dict['precip_mean'] + predictors_dict['temp_mean']
predictors

['precipitation_1.0_mean',
 'precipitation_2.0_mean',
 'precipitation_3.0_mean',
 'precipitation_4.0_mean',
 'precipitation_5.0_mean',
 'precipitation_6.0_mean',
 'precipitation_7.0_mean',
 'precipitation_8.0_mean',
 'precipitation_9.0_mean',
 'precipitation_10.0_mean',
 'precipitation_11.0_mean',
 'precipitation_12.0_mean',
 'air_temperature_1.0_mean',
 'air_temperature_2.0_mean',
 'air_temperature_3.0_mean',
 'air_temperature_4.0_mean',
 'air_temperature_5.0_mean',
 'air_temperature_6.0_mean',
 'air_temperature_7.0_mean',
 'air_temperature_8.0_mean',
 'air_temperature_9.0_mean',
 'air_temperature_10.0_mean',
 'air_temperature_11.0_mean',
 'air_temperature_12.0_mean']

#### Train/test split

In [36]:
random_seed = tutorial_config['random_seed']

In [37]:
test_frac = 0.1
val_frac = 0.1
val_frac_sub = (val_frac / (1.0-test_frac) )

In [38]:
test_df = zones_df.groupby(['period_start','scenario']).sample(frac=test_frac, random_state=random_seed)
remain_df = zones_df.drop(test_df.index)

In [39]:
val_df = remain_df.groupby(['period_start','scenario']).sample(frac=val_frac_sub, random_state=random_seed)
train_df = remain_df.drop(val_df.index)


#### Data Preparation


,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [41]:
X_train = input_scaler.transform(train_df[predictors])
X_val = input_scaler.transform(val_df[predictors])
X_test = input_scaler.transform(test_df[predictors])

In [42]:
train_df[[target_var]].value_counts()

climate_group
E                116146
D                 80766
B                 60790
A                 40252
C                 27252
Name: count, dtype: int64

In [43]:
target_encoder = sklearn.preprocessing.LabelEncoder()


/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:103: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


LabelEncoder()

In [44]:
y_train = target_encoder.transform(train_df[[target_var]])
y_val = target_encoder.transform(val_df[[target_var]])
y_test = target_encoder.transform(test_df[[target_var]])


/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)
/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dty

## Using Pytorch

From this point in the tutorial, we diverge from what was done in the ml training pipeline tutorial as instead of setting up and training our model in scikit-learn, we're going to use a more sophisticated machine learning library called pytorch. This gives us more more control over how we implement our neural network and gives us much more power to use advanced architectures, losss functions and distributed computing techniques.

#### Training hyperparameters

In [154]:
batch_size=8
num_epochs = 10

### Define a data loader

Further reading
- https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html
- https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
- https://www.geeksforgeeks.org/deep-learning/converting-a-pandas-dataframe-to-a-pytorch-tensor/
- https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelBinarizer.html

In [136]:
class ClimateZonesDataset(torch.utils.data.Dataset):
    """
    Inspired by this tutorial:
    https://machinelearningmastery.com/converting-pandas-dataframes-to-pytorch-dataloaders-for-custom-deep-learning-model-training/
    """
    def __init__(self, df_ml, predictor_features, target_feature, stats_dict=None):
        self._df_ml = df_ml.reset_index().drop(['index'],axis='columns')

        self.input_scaler = sklearn.preprocessing.StandardScaler()
        if stats_dict is None:
            self.input_scaler.fit(self._df_ml[predictor_features])
        else:
            self.input_scaler.mean_ = numpy.array(stats_dict['input_mean'])
            self.input_scaler.scale_ = numpy.array(stats_dict['input_scale'])

        self._X = torch.tensor(self.input_scaler.transform(self._df_ml[predictor_features]),  
                               dtype=torch.float32)


        self.target_encoder = sklearn.preprocessing.LabelBinarizer(sparse_output=False)
        if stats_dict is None:
            self.target_encoder.fit(self._df_ml[[target_feature]])
        else:
            self.target_encoder.classes_ = numpy.array(stats_dict['target_classes'],dtype='object')

        
        self._y = torch.tensor(self.target_encoder.transform(self._df_ml[[target_feature]]),
                               dtype=torch.float32)

        self.stats_dict = {
            'input_mean': [float(v1) for v1 in self.input_scaler.mean_],
            'input_scale': [float(v1) for v1 in self.input_scaler.scale_],
            'target_classes': list(self.target_encoder.classes_),
        }
        
    def _repr_html_(self):
        return f'''
        <h1>Climate Zones Dataset</h1>
        Number of samples {len(self._X)}
        '''
    
    def __len__(self):
        return len(self._X)

    def __getitem__(self,idx):
        return self._X[idx], self._y[idx]


        

In [137]:
cz_train_ds = ClimateZonesDataset(train_df, predictors, target_var)
cz_train_ds

We now intialise the validate and test set data loaders. Note that we initialise the preprocessing objects with the values learned from the training data, rather than calculating them on the validate or test data.

In [138]:
cz_val_ds = ClimateZonesDataset(val_df, predictors, target_var, cz_train_ds.stats_dict)
cz_test_ds = ClimateZonesDataset(test_df, predictors, target_var, cz_train_ds.stats_dict)

/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/home/users/shaddad/mohc_shared/users/shaddad/venv/ai4c_nb_cpu/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [139]:
cz_train_ds[123]

(tensor([-0.4307, -0.4175, -0.4748, -0.4082, -0.5822, -0.5345, -0.4380, -0.4644,
         -0.3433, -0.4762, -0.3574, -0.4099, -1.1763, -1.0522, -0.9483, -0.6865,
         -0.3488, -0.1132, -0.0330, -0.0719, -0.3299, -0.6610, -1.0060, -1.1405]),
 tensor([0., 0., 0., 0., 1.]))

In [140]:
cz_val_ds[10:15]

(tensor([[-0.4408, -0.3281, -0.1737, -0.2187, -0.2642, -0.3904, -0.3665, -0.3730,
          -0.4555, -0.4177, -0.4437, -0.4373, -0.5886, -0.7146, -0.8674, -0.9889,
          -1.0984, -1.1859, -1.2289, -1.2352, -1.1710, -1.0693, -0.9048, -0.6668],
         [-0.5651, -0.5227, -0.5370, -0.5412, -0.5645, -0.5760, -0.5667, -0.5496,
          -0.6736, -0.6741, -0.6055, -0.5566, -0.9222, -1.0107, -1.1763, -1.3347,
          -1.4611, -1.4826, -1.4871, -1.4774, -1.4484, -1.4140, -1.2316, -0.9931],
         [-0.5389, -0.5087, -0.4206, -0.4052, -0.3257, -0.5483, -0.5183, -0.5140,
          -0.5723, -0.5571, -0.6006, -0.5566, -0.2212, -0.4738, -0.7350, -0.8927,
          -1.0188, -1.0741, -1.1504, -1.2124, -1.1016, -0.9153, -0.5810, -0.2653],
         [-0.5786, -0.5990, -0.6152, -0.6296, -0.6531, -0.6305, -0.6151, -0.6093,
          -0.7026, -0.7228, -0.6742, -0.6034, -1.1181, -1.2958, -1.5538, -1.7585,
          -1.8879, -1.8760, -1.8485, -1.8251, -1.7909, -1.7417, -1.5150, -1.1970],
         [-0

In [84]:
cz_train_loader = torch.utils.data.DataLoader(
        cz_train_ds, batch_size=batch_size, shuffle=True, num_workers=1,
    )
cz_val_loader = torch.utils.data.DataLoader(
        cz_val_ds, batch_size=batch_size, shuffle=False, num_workers=1,
    )

In [86]:
count = 0
for i1 in cz_train_loader:
    print(i1)
    count +=1
    if count > 5:
        break

[tensor([[-1.6196e-01, -5.7226e-02, -1.3287e-01, -3.3530e-01, -5.2997e-01,
         -6.5491e-01, -7.1055e-01, -7.5412e-01, -6.7995e-01, -3.4266e-01,
         -5.1478e-02, -9.4512e-02,  9.0829e-01,  9.3217e-01,  9.1481e-01,
          8.2978e-01,  7.8560e-01,  7.5749e-01,  7.8699e-01,  8.2236e-01,
          8.4707e-01,  9.2579e-01,  9.6604e-01,  9.3939e-01],
        [-2.2957e-01, -2.9460e-01, -3.5759e-01, -2.0610e-01, -3.0995e-03,
          1.3618e-01,  9.1863e-02,  3.0618e-02,  2.4980e-02,  7.8398e-02,
          9.1697e-02, -6.2706e-02, -4.7838e-01, -2.6069e-01,  7.8858e-02,
          3.0068e-01,  4.6275e-01,  6.0700e-01,  6.4035e-01,  5.5948e-01,
          3.9837e-01,  2.1675e-01, -1.4144e-01, -4.3466e-01],
        [ 3.1169e+00,  4.7898e+00,  5.1351e+00,  4.5353e+00,  2.3235e+00,
          2.8674e-01, -1.6187e-01, -4.2179e-01, -3.1255e-01,  2.2752e-01,
          6.3399e-01,  1.5072e+00,  1.4776e+00,  1.3861e+00,  1.2286e+00,
          1.0771e+00,  9.4703e-01,  8.3273e-01,  7.7254e-01, 

In [145]:
X_train.shape

(325206, 24)

In [157]:
class ClimateZoneClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = torch.nn.Linear(24, 60)
        self.act1 = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(60, 60)
        self.act2 = torch.nn.ReLU()
        self.layer3 = torch.nn.Linear(60, 60)
        self.act3 = torch.nn.ReLU()
        self.output = torch.nn.Linear(60, 5)
        self.sigmoid = torch.nn.Sigmoid()
 
    def forward(self, x):
        x = self.act1(self.layer1(x))
        x = self.act2(self.layer2(x))
        x = self.act3(self.layer3(x))
        x = self.sigmoid(self.output(x))
        return x
    


In [158]:
cz_classifier = ClimateZoneClassifier()

In [159]:
 
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cz_classifier.parameters(), lr=0.001)

In [160]:
for epoch in range(num_epochs):
    cz_classifier.train()
    for batch_X, batch_y in cz_train_loader:
        optimizer.zero_grad()
        predictions = cz_classifier(batch_X)
        loss = loss_fn(predictions, batch_y)
        loss.backward()
        optimizer.step()
        print(loss)


RuntimeError: expected scalar type Long but found Float

ClimateZoneClassifier(
  (layer1): Linear(in_features=12, out_features=60, bias=True)
  (act1): ReLU()
  (layer2): Linear(in_features=60, out_features=60, bias=True)
  (act2): ReLU()
  (layer3): Linear(in_features=60, out_features=60, bias=True)
  (act3): ReLU()
  (output): Linear(in_features=60, out_features=5, bias=True)
  (sigmoid): Sigmoid()
)

## 9. Evaluation
Having trained a model, we want to evaluate how well it predicts the target values. Doing this for the training set is a god starting point for understanding how well the relationships present in the training data have been learnt. To see how well this mapping represents the general relationships of interests, rather than being specific to the sample of data present in the trianing set, the more important result is the metric scores for the validation set.

We will investigate further in the evaluation notebook. 

In [45]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_train, y_pred_train[clf_name])

In [46]:
for clf_name, clf_obj in classifiers_dict.items():
    sklearn.metrics.precision_recall_fscore_support(y_val, y_pred_val[clf_name])

We will go into more details on evaluating the performance of a machine learning model in a separate notebook.

### Loading and storing with ML Flow
One additional feature of usingn ML flow for tracking our experiments, is that the trained model is saved as a part of the 

# Exercises

Once you have worked through the tutorial above, you can test your knowledge by adapting the code in the tutorial to experiment with different options for training the model and comparing results.

### Historic and future comparison
So far we have put all the data from historic and future scenarios together into one big dataset. The mapping from climate variables to climate zones should be the same in the future, even as the climate zones of particular places changes. We can test that by using historic data as our training set and then use future scenarios as our test data.  

In [ ]:
# insert  code here

### Divide train/test by geographic region
Doing our train/test split randomly, we are likely to have lots of correlations between examples in the train/val/test sets. One way we could reduce this is by spliting by geographic region. As the zones are highly corrlated with latitude, we couldn't use latitude for splitting. Instead we could divide by longitude. We would need to ensure all zones are represented in train, val and test sets.

In [ ]:
# insert  code here

### Addressing class imbalance
We have seen that the number of memebers of different classes 

In [ ]:
# insert  code here

In [ ]:
# train on historic data and predict on future data

In [ ]:
# try to divide train/test by geographic region. Use longitude 

In [ ]:
# try to balance classes in train set and compare results

### Work with More classes or higher resolution 


In [ ]:
# Try to load the 0.5 degree data and work with that

In [ ]:
# try to create a classifier for all 30 climate subgroups as the target

### Next steps or potential follow on material

Additional excercises in this tutorial material includes:
- Training a neural network using pytorch
- Training a CNN autoencoder on CMIP6 data
- Training a RNN on Jena weather dataset for time series prediction. 


### Data statement
This data used in this notebook is derived from the Koppen-Geiger Climate Cliassfication dataset created by GloH2O.

###     References
- [Climate Zones Dataset](https://www.gloh2o.org/koppen/#:~:text=The%20K%C3%B6ppen%2DGeiger%20climate%20classification%20maps%20are%20high%2Dresolution,Climate%20Sensitivity%20(ECS)%2C%20and%20historical%20warming%20trend)
